In [1]:
import subprocess
import json
import pandas as pd
import os
import google.generativeai as genai
from dotenv import load_dotenv
import re
import numpy as np

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

c:\Users\alex9\Documents\med_agent\key_words_coscience\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

def compare_results(original: str, generated: str, embedding_function) -> str:
    original_embedding = embedding_function(original)
    generated_embedding = embedding_function(generated)
    
    original_embedding = [original_embedding]
    generated_embedding = [generated_embedding]
    return float(cosine_similarity(original_embedding, generated_embedding)[0, 0])

In [3]:
def gemini_embedding(text: str):
    response = genai.embed_content(
        model="gemini-embedding-001",
        content=text,
        task_type="retrieval_query"
    )
    return response['embedding']

In [4]:
def ollama_query_model(prompt: str, model_name='llama3.1:8b') -> str:
    cmd = ["ollama", "run", model_name]
    process = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )
    stdout, stderr = process.communicate(prompt, timeout=60)

    responses = []
    for line in stdout.splitlines():
        try:
            data = json.loads(line)
            if "response" in data:
                responses.append(data["response"])
        except json.JSONDecodeError:
            responses.append(line.strip())

    text = " ".join(responses).strip()
    if not text:
        text = stdout.strip()

    return text



prompts

In [5]:
PROMPT_1 = """
Ты — экспертная система для анализа клинических исследовательских гипотез.
Проанализируй приведённую гипотезу и оцени, можно ли считать её логически и научно обоснованной.
Ответ дай строго в формате:
"утвердительно" или "отрицательно". После этого — одно краткое причинное предложение.

Гипотеза: {hypothesis}
"""

PROMPT_2 = """
Выдели из текста до пяти ключевых научных или медицинских концепций — 
существенных для понимания сути гипотезы (например: "инсулинорезистентность", "воспалительный ответ").

Ответ представь в виде списка, разделённого запятыми, без нумерации, без пояснений.

Текст: "{hypothesis}"
"""

PROMPT_3 = """
Для каждой ключевой концепции найди связанные с ней слова или выражения из текста — 
включая синонимы, уточняющие термины и контекстные признаки.

Результат представь в виде корректного Python-словаря (dict), где:
ключ — концепция,
значение — список связанных слов.
Без форматирования.
Без форматирования в формате markdown.
Не добавляй ничего лишнего, только словарь.
Ключевые концепции: {key_concepts}
Текст: "{hypothesis}"
"""

PROMPT_5 = """
Восстанови замаскированные (пропущенные) слова или выражения в тексте. 
Если возможно несколько вариантов — выбери наиболее обобщённый и нейтральный.
Заполняй пропуски, начиная с последнего, чтобы сохранить контекст.

Выведи ПОЛНЫЙ восстановленный текст.
Не добавляй ничего лишнего, только восстановленный текст.
Текст для анализа:
{masked_text}
"""



In [6]:
def process_list(text: str) -> list:
    return [item.strip() for item in text.split(",") if item.strip()]

In [7]:
import ast

def process_string_to_dict(text: str) -> dict:
    start_index = text.find('{')
    end_index = text.rfind('}') + 1

    dict_str = text[start_index:end_index].strip()

    return ast.literal_eval(dict_str)

In [ ]:
def mask_text(text: str, words_to_mask: list) -> str:
    """
    Заменяет все слова из списка на [MASKED] в исходном тексте
    """
    pattern = r'\b(' + '|'.join(map(re.escape, words_to_mask)) + r')\b'
    
    masked_text = re.sub(pattern, '[MASKED]', text, flags=re.IGNORECASE)
    return masked_text

In [9]:
def run_pipeline(hypothesis: str, model_query_func, embedding_function=gemini_embedding) -> dict:
    results = {}

    results["understanding"] = model_query_func(PROMPT_1.format(hypothesis=hypothesis))

    key_concepts = model_query_func(PROMPT_2.format(hypothesis=hypothesis))
    
    results["key_concepts"] = process_list(key_concepts)

    text_dict = model_query_func(PROMPT_3.format(hypothesis=hypothesis, key_concepts=key_concepts))
    results["dictionary"] = process_string_to_dict(text_dict)

    masked_texts = [mask_text(text=hypothesis, words_to_mask = x) for x in list(results["dictionary"].values()) if x]
    results["masked_text"] = masked_texts

    alt_answers = []
    for mtext in masked_texts:
        alt_answers.append(model_query_func(PROMPT_5.format(masked_text=mtext)))
    results["alt_answer"] = alt_answers
    results["comparison"] = []
    for answer in alt_answers:
        results["comparison"].append(compare_results(hypothesis, answer, embedding_function))

    return results


# Прогон всех гипотез через локальную модель (llama3.1:8b)

In [12]:
df = pd.read_excel("data/datanoprompts.xlsx")

understanding_list = []
key_concepts_list = []
dictionary_list = []
masked_text_list = []
alt_answer_list = []
comparison_list = []

for hypothesis in df['Гипотеза']:
    result = run_pipeline(hypothesis, model_query_func=ollama_query_model, embedding_function=gemini_embedding)
    print(hypothesis)
    understanding_list.append(result["understanding"])
    key_concepts_list.append(result["key_concepts"])
    dictionary_list.append(result["dictionary"])
    masked_text_list.append(result["masked_text"])
    alt_answer_list.append(result["alt_answer"])
    comparison_list.append(result["comparison"])

df["understanding"] = understanding_list
df["key_concepts"] = key_concepts_list
df["dictionary"] = dictionary_list
df["masked_text"] = masked_text_list
df["alt_answer"] = alt_answer_list
df["comparison"] = comparison_list

df.to_csv("data/hypotheses_processed.csv", index=False, encoding="utf-8-sig")



Транскатетерный низкодозовый тромболизис у пациентов с тромбоэмболией легочной артерии промежуточно-высокого риска снижает годичную летальность по любой причине.
Прием никорандила в дозе 20 мг 2 раза в сутки на фоне применения фторпиримидинов в пациентов с колоректальным раком приводит к снижению частоты развития острого коронарного синдрома.
Внедрение дистанционного мониторинга уровня артериального давления и пульса на фоне таргетной терапии бевацизумабом у пациентов с солидными опухолями уменьшает частоту сердечно-сосудистых осложнений на фоне этого лечения.
Первичная профилактика венозного тромбоза после оперативного лечения глиом уменьшит в два раза число венозных тромбоэмболических событий на этапе госпитализации без значимого увеличения числа больших кровотечений
Реперфузионное лечение у пациентов с тромбоэмболией легочной артерии высокого и промежуточного риска тридцати дневной летальности снижает риск развития посттромбоэмболического синдрома
Терапия ингибиторами контрольных то

In [14]:
df_processed = pd.read_csv("data/hypotheses_processed.csv")
df_processed

,Гипотеза,Структура запроса,understanding,key_concepts,dictionary,masked_text,alt_answer,comparison
0,Транскатетерный низкодозовый тромболизис у пац...,1.Популяция: пациенты с тромбоэмболией промежу...,Ответ: утвердительно. Причина: Гипотеза основ...,"['инсулинорезистентность', 'воспалительный отв...","{'инсулинорезистентность': [], 'воспалительный...",['[MASKED] [MASKED] тромболизис у пациентов с ...,['РЕЗУЛЬТАТ: Тромболизис промежуточной дозы у ...,"[0.9248742870032387, 0.5158079373771689, 0.624..."
1,Прием никорандила в дозе 20 мг 2 раза в сутки ...,1.Популяция: пациенты с колоректальным раком\n...,Ответ: утвердительно. Причина: Гипотеза обосн...,"['инсулинорезистентность', 'воспалительный отв...","{'инсулинорезистентность': [], 'воспалительный...",['Прием никорандила в дозе 20 мг 2 раза в сутк...,['ПОЛНЫЙ восстановленный текст: Прием никоран...,"[0.9843317282269907, 0.984274476275922]"
2,Внедрение дистанционного мониторинга уровня ар...,1.Популяция: пациенты с колоректальным раком\n...,Ответ: утвердительно. Причинное предложение: ...,"['Бевацизумаб', 'артериальное давление', 'пуль...","{'Бевацизумаб': ['таргетная терапия', 'пациент...",['Внедрение дистанционного мониторинга уровня ...,['Восстановленный текст: Внедрение дистанционн...,"[0.9815016489298694, 0.9587560486796212, 0.986..."
3,Первичная профилактика венозного тромбоза посл...,1.Популяция: пациенты с глиомами\n2.Вмешательс...,Ответ: утвердительно. Гипотеза основана на пон...,"['Венозный тромбоз', 'профилактика', 'оператив...",{'Венозный тромбоз': ['венозные тромбоэмболиче...,['Первичная профилактика венозного тромбоза по...,['Восстановленный текст: Первичная профилактик...,"[0.9725880057412093, 0.914485431065898, 0.9748..."
4,Реперфузионное лечение у пациентов с тромбоэмб...,1.Популяция: пациенты с тромбоэмболией легочно...,"Ответ: ""утвердительно"". Причинное предложение...","['Тромбоэмболический синдром', 'Реперфузионное...","{'Тромбоэмболический синдром': ['тромбоз', 'эм...",['Реперфузионное лечение у пациентов с тромбоэ...,['Полный восстановленный текст: Реперфузионное...,"[0.9784318824593189, 0.9570146311625769, 0.973..."
5,Терапия ингибиторами контрольных точек иммунно...,1.Популяция: пациенты с солидными опухолями ни...,Ответ: отрицательно Причина: гипотеза не имее...,"['иммунный ответ', 'контрольные точки иммунног...",{'иммунный ответ': ['контрольные точки иммунно...,['Терапия ингибиторами контрольных точек иммун...,['Восстановленный текст: Терапия ингибиторами ...,"[0.9704685039182558, 0.9832951838360696, 0.921..."
6,Анемия является независимым фактором риска лет...,1.Популяция: пациенту с легочной гипертензией\...,"Ответ: утвердительно. Причинами того, что гип...","['Анемия', 'легочная гипертензия', 'факторы ри...",{'Анемия': ['независимый фактор риска летально...,['Анемия является независимым фактором риска л...,['Я не смогу помочь вам в выполнении этого зад...,"[0.4862097459708545, 0.8317877567520655, 0.863..."
7,Разработка мультимодального диагностического п...,1.Популяция: пациенты с подозрением на тромбоз...,Ответ: утвердительно. Причина: Присутствуют ко...,"['саркома легочной артерии', 'тромбоз легочной...",{'саркома легочной артерии': ['легочная сарком...,['Разработка мультимодального диагностического...,['Восстановленный текст: Разработка мультимода...,"[0.9831029353270099, 0.9765904605656102, 0.990..."
8,Разработка метода ретроградной реканализации ...,1.Популяция: пациенты ХТЭЛГ с окклюзией сегмен...,Отрицательно. Гипотеза не содержит четкого опр...,"['Легочная гипертензия', 'окклюзия легочной ар...",{'Легочная гипертензия': ['хроническая тромбоэ...,[' Разработка метода ретроградной реканализаци...,['Восстановленный текст: Разработка метода рет...,"[0.9800332032619259, 0.9800332032619259, 0.953..."
9,Комбинация хирургической резекции с адъювантно...,1.Популяция: пациенты с первичной саркомой лег...,Ответ: утвердительно. Причина: Гипотеза основа...,['Правило разделения на ключевые понятия не пр...,"{'сарcoma': ['легочная артерия', 'легочный'], .

# Тест с google ai studio

In [15]:
def gemini_query_model(prompt: str, model_name='models/gemini-2.5-flash-lite') -> str:
    """
    Запрос к модели Google AI Studio (Gemini) вместо Ollama.
    """
    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)

        if hasattr(response, "text"):
            return response.text.strip()
        elif hasattr(response, "candidates") and len(response.candidates) > 0:
            return response.candidates[0].content.parts[0].text.strip()
        else:
            return str(response).strip()

    except Exception as e:
        print(f"Ошибка при обращении к Google AI Studio: {e}")
        return ""

In [16]:
hypothesis = "Транскатетерный низкодозовый тромболизис у пациентов с тромбоэмболией легочной артерии промежуточно-высокого риска снижает годичную летальность по любой причине."

result = run_pipeline(hypothesis, model_query_func=gemini_query_model, embedding_function=gemini_embedding)

In [17]:
result

{'understanding': 'утвердительно. Гипотеза логически обоснована, так как тромболизис направлен на устранение основной причины эмболии, что потенциально может снизить смертность.',
 'key_concepts': ['транскатетерный низкодозовый тромболизис',
  'тромбоэмболия легочной артерии',
  'промежуточно-высокий риск',
  'годичная летальность',
  'любая причина'],
 'dictionary': {'транскатетерный низкодозовый тромболизис': ['транскатетерный низкодозовый тромболизис'],
  'тромбоэмболия легочной артерии': ['тромбоэмболия легочной артерии', 'тэла'],
  'промежуточно-высокий риск': ['промежуточно-высокого риска'],
  'годичная летальность': ['годичную летальность'],
  'любая причина': ['любой причине']},
 'masked_text': ['[MASKED] у пациентов с тромбоэмболией легочной артерии промежуточно-высокого риска снижает годичную летальность по любой причине.',
  'Транскатетерный низкодозовый тромболизис у пациентов с тромбоэмболией легочной артерии промежуточно-высокого риска снижает годичную летальность по любо

In [18]:
hypothesis2 = "Прием никорандила в дозе 20 мг 2 раза в сутки на фоне применения фторпиримидинов в пациентов с колоректальным раком приводит к снижению частоты развития острого коронарного синдрома."

result2 = run_pipeline(hypothesis, model_query_func=gemini_query_model, embedding_function=gemini_embedding)

In [19]:
result2

{'understanding': 'утвердительно. Гипотеза логична, так как тромболизис направлен на растворение тромба, что потенциально может улучшить прогноз и снизить смертность.',
 'key_concepts': ['транскатетерный низкодозовый тромболизис',
  'тромбоэмболия легочной артерии',
  'промежуточно-высокий риск',
  'годичная летальность',
  'любая причина'],
 'dictionary': {'транскатетерный низкодозовый тромболизис': ['транскатетерный низкодозовый тромболизис',
   'тромболизис'],
  'тромбоэмболия легочной артерии': ['тромбоэмболия легочной артерии',
   'тромбоэмболия'],
  'промежуточно-высокий риск': ['промежуточно-высокий риск'],
  'годичная летальность': ['годичная летальность', 'летальность'],
  'любая причина': ['любой причине', 'по любой причине']},
 'masked_text': ['[MASKED] у пациентов с тромбоэмболией легочной артерии промежуточно-высокого риска снижает годичную летальность по любой причине.',
  'Транскатетерный низкодозовый тромболизис у пациентов с тромбоэмболией легочной артерии промежуточно